# This notebook calculate the ratio of shear stress to effective normal stress, which will be used as continuous responses in the subsequent DGSA sensitivity analysis

# 3. Calculate the ratio of shear stress to effective normal stress

## perform fault slip analysis on each case

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from fault_slip_analysis import tau_sigma_ratio
from tqdm import tqdm

# set up path
base_path = Path('.')
# user inputs
name_prefix = '250922'; n_cases = 90; n_faults = 12
ratio_by_fault = np.full((107,117,5,6), np.nan)
fault_info = pd.read_csv(base_path/'data'/'raw'/'fault_strike_dip.csv')
coor_fault = np.load(base_path/'data'/'coor_fault'/'JD_Sula_2025_gmc_coor&fault_reservoir.npy')
parameters = pd.read_csv(base_path/'data'/'params_responses'/f'{name_prefix}_CMG_parameters.csv')

save_folder_path = base_path/'data'/f'{name_prefix}_ratio'
save_folder_path.mkdir(parents=True, exist_ok=True)

# run analysis
# for case_num in tqdm(range(50,51), desc='Running stress-based fault slip analysis'):
for case_num in tqdm(range(1,n_cases+1), desc='Calculating the ratio of shear stress to effective normal stress'):
    # load principal stress arrays
    SH = np.load(base_path/'data'/f"{name_prefix}_gmc"/f'case{case_num}_STRESMXP.npy')
    Sh = np.load(base_path/'data'/f"{name_prefix}_gmc"/f'case{case_num}_STRESMNP.npy')
    Sv = np.load(base_path/'data'/f"{name_prefix}_gmc"/f'case{case_num}_STRESINT.npy')
    # extract the azimuth of the maximum horizontal stress from the parameter dataframe
    SH_azi = parameters.loc[parameters["case_name"] == f'case{case_num}','SH_azi_deg'].iloc[0]

    for fault_id in range(n_faults):
        row = fault_info.loc[fault_info['fault_id'] == fault_id].iloc[0]
        fault_slip = tau_sigma_ratio(
            SH = SH,
            Sh = Sh,
            Sv = Sv,
            SH_azi = SH_azi,
            fault_strike = row['fault_strike_deg'],
            fault_dip = row['dip_angle_deg']
            )

        # save analysis to the specific fault 
        fault_id_mask = (coor_fault[:,:,:,3] == fault_id)
        ratio_by_fault[fault_id_mask] = fault_slip[fault_id_mask]

    # save results for one case
    np.save(save_folder_path/f'case{case_num}_ratio.npy', ratio_by_fault)

print("\nFinished analyzing all cases.")

Calculating the ratio of shear stress to effective normal stress: 100%|██████████| 90/90 [02:17<00:00,  1.52s/it]


Finished analyzing all cases.


In [6]:
arr = np.load('data/250922_ratio/case10_ratio.npy')
arr.shape
print(np.nanmean(arr))
print(np.nanmax(arr))

0.21602929530127218
0.5151796395298871


## combine all cases in a numpy array (n_cases,n_faults,n_times) containing total number of slipped cells

In [8]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# user inputs
name_prefix = '250922'; n_cases = 90; n_faults = 12; n_times = 6
fault_info = pd.read_csv('data/raw/fault_strike_dip.csv')
coor_fault = np.load('data/coor_fault/JD_Sula_2025_gmc_coor&fault_reservoir.npy')
ratio_combined = np.full((n_cases,n_faults,n_times), np.nan)

# set up paths
base_path = Path('.')
ratio_folder = base_path/'data'/f'{name_prefix}_ratio'

for case_num in tqdm(range(1,n_cases+1), desc='Combining FSA results for all cases'):
    FSA = np.load(ratio_folder/f'case{case_num}_ratio.npy')

    for fault_id in range(0,n_faults):
        # save analysis to the specific fault 
        fault_id_mask = (coor_fault[:,:,:,3] == fault_id)
        ratio_combined[case_num-1,fault_id,:] = np.nanmean(FSA[fault_id_mask],axis=0)

# save
np.save(base_path/'data'/f'{name_prefix}_ratio_combined.npy',ratio_combined)

print("\nFinished combining FSA results for all cases to one numpy array.")
# np.savetxt(base_path/'data'/f'{save_file_prefix}_FSA_combined.csv',FSA_combined,delimiter=",",fmt="%.4f")

Combining FSA results for all cases: 100%|██████████| 90/90 [00:00<00:00, 307.35it/s]


Finished combining FSA results for all cases to one numpy array.


In [9]:
print(ratio_combined.shape)
print(ratio_combined[4,:,:])

(90, 12, 6)
[[0.3489467  0.3600442  0.37196105 0.37196897 0.37105492 0.37066178]
 [0.05521347 0.05740173 0.06015587 0.0605021  0.06028777 0.06019497]
 [0.00400915 0.00410412 0.00419705 0.00419051 0.00418346 0.00418044]
 [0.0185447  0.01897718 0.01933883 0.01925356 0.01923422 0.01922244]
 [0.48609169 0.50538095 0.52214189 0.51820395 0.5174134  0.51686432]
 [0.11156493 0.1146517  0.11841432 0.1188726  0.11857113 0.11844668]
 [0.52376455 0.54103758 0.56297157 0.56597305 0.56452318 0.56378907]
 [0.03099596 0.03165433 0.03232501 0.03230352 0.03226679 0.03224488]
 [0.01953335 0.02048075 0.02174593 0.02194912 0.02185246 0.0218091 ]
 [0.0550981  0.05643028 0.0575007  0.05719902 0.05714681 0.05711188]
 [0.09901805 0.10116208 0.10344036 0.10346841 0.10328552 0.10321124]
 [0.29558033 0.30447535 0.31360691 0.31319197 0.31253918 0.31224026]]


# calculate responses for DGSA sensitivity analysis

per fault

In [12]:
import numpy as np

name_prefix = '250922'; fault_id = [0,4,6]; year = 2050; year_list = [2030, 2040, 2050, 2060, 2550, 3050]
ratio_combined = np.load(f'data/{name_prefix}_ratio_combined.npy')
print(ratio_combined.shape)
resp = ratio_combined[:,fault_id,year_list.index(year)]
# np.savetxt(f'data/params_responses/{name_prefix}_CMG_responses_fault{fault_id}.csv',resp,delimiter=',',fmt='%d',header='FSA',comments='')
np.savetxt(f'data/params_responses/{name_prefix}_CMG_responses_ratio.csv',resp,delimiter=',',fmt='%.3f',header='FSA',comments='')
resp

(90, 12, 6)


array([[0.42402952, 0.45329523, 0.49927077],
       [0.46425411, 0.41674959, 0.46374836],
       [0.49634135, 0.30918657, 0.35668433],
       [0.4095379 , 0.4291247 , 0.47253721],
       [0.37196105, 0.52214189, 0.56297157],
       [0.35749906, 0.50590651, 0.54624735],
       [0.49962962, 0.36481081, 0.41489094],
       [0.35497187, 0.49627458, 0.53336496],
       [0.39496671, 0.43612526, 0.47626412],
       [0.44004239, 0.43791197, 0.48418891],
       [0.49096081, 0.36442212, 0.41505684],
       [0.50381932, 0.28987482, 0.33848105],
       [0.50795511, 0.39807516, 0.45166897],
       [0.41871642, 0.35748312, 0.39884219],
       [0.33927515, 0.44516225, 0.47448708],
       [0.39728606, 0.42916485, 0.46958578],
       [0.49298806, 0.38514131, 0.43714185],
       [0.48104118, 0.48358   , 0.53895419],
       [0.49621724, 0.30240739, 0.35066631],
       [0.47203135, 0.27396226, 0.31933091],
       [0.35640817, 0.45493973, 0.48868223],
       [0.46978997, 0.29252943, 0.33973421],
       [0.

all faults added

In [41]:
import numpy as np

name_prefix = '250922'
FSA_combined = np.load(f'data/{name_prefix}_FSA_combined.npy')
print(FSA_combined.shape)
resp = np.sum(FSA_combined,axis=1).astype(int)
# resp[resp != 0] = 1
# np.savetxt(f'data/params_responses/{name_prefix}_CMG_responses_fault{fault_id}.csv',resp,delimiter=',',fmt='%d',header='FSA',comments='')
resp

(90, 12, 6)


array([[  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   2,   2,   2],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,   0,   0,   0,   0],
       [  0,   0,  79, 13